# StyleSense: Real-Time AI Fashion Recommendations using MobileNet

ISPCC 2025 — MobileNetV2-based fashion aesthetic classification into 5 social categories:
- Business, Casual, Night Party, Sports, Wedding

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print(f"TensorFlow: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
from src.model import build_stylesense_model, compile_model, get_callbacks
from src.preprocess import create_data_generators
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_auc, plot_training_history, plot_prediction_confidence
import config

In [ ]:
train_gen, val_gen = create_data_generators(batch_size=32)
print(f"Classes: {train_gen.class_indices}")
print(f"Train: {train_gen.samples} images")
print(f"Val:   {val_gen.samples} images")

In [ ]:
model = build_stylesense_model(trainable_base=False)
model = compile_model(model, learning_rate=0.001)
model.summary()

In [ ]:
callbacks = get_callbacks()
history = model.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
results = evaluate_model(model, val_gen)

In [ ]:
plot_training_history(history, save_path_prefix='stylesense')
plot_confusion_matrix(results['confusion_matrix'], results['class_labels'], save_path='stylesense_confusion_matrix.png')
plot_roc_auc(results['y_true'], results['predictions'], results['class_labels'], save_path='stylesense_roc_auc.png')
plot_prediction_confidence(results['predictions'], results['y_true'], results['class_labels'], save_path='stylesense_confidence_dist.png')

In [ ]:
model.save('saved_models/stylesense_final.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('tflite/stylesense_model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite size: {len(tflite_model) / (1024*1024):.2f} MB')

In [ ]:
from src.predict import StyleSensePredictor
predictor = StyleSensePredictor(model_path='saved_models/stylesense_final.keras')
result = predictor.predict('data/raw/Casual/sample.jpg')  # replace with your image
print(f"Predicted: {result['predicted_class']} ({result['confidence']*100:.2f}%)")